# 01 · Library EDA — what taste looks like as data

Our raw material for building playlists is the user's **own saved playlists**.
Before modelling anything, we look at the library as data: how big are the
playlists, which artists dominate, and which tracks act as connective tissue
across many playlists.

Grounding: [`research/05-inferring-taste-from-libraries.md`](../../research/05-inferring-taste-from-libraries.md).

## Setup
Connect to the SQLite DB produced by `build_db.py`.

In [ ]:
import sqlite3
import pandas as pd

DB = '../playlists.db'  # run build_db.py first if this is missing
con = sqlite3.connect(DB)
def q(sql, **kw):
    return pd.read_sql_query(sql, con, params=kw)

q('SELECT COUNT(*) AS playlists FROM playlists')

## How big is the library, and how big are playlists?

In [ ]:
overview = q('''
  SELECT (SELECT COUNT(*) FROM playlists) AS playlists,
         (SELECT COUNT(*) FROM tracks) AS unique_tracks,
         (SELECT COUNT(*) FROM playlist_tracks) AS memberships
''')
overview

In [ ]:
sizes = q('SELECT name, track_count FROM playlists ORDER BY track_count DESC')
display(sizes.head(10))
sizes['track_count'].describe()

Playlist-size distribution — most playlists are curated (tens of tracks),
with a few large 'dump' playlists (Liked Songs) that we treat differently
for co-occurrence (`cooccurrence_included = 0`).

In [ ]:
ax = sizes['track_count'].plot(kind='hist', bins=30, title='Playlist size distribution')
ax.set_xlabel('tracks per playlist')

## Who dominates? Top artists by track count
A first-order taste signal.

In [ ]:
top_artists = q('''
  SELECT artist, COUNT(*) AS tracks
  FROM tracks GROUP BY artist ORDER BY tracks DESC LIMIT 20''')
top_artists.plot(kind='barh', x='artist', y='tracks', figsize=(6,7),
                 title='Top 20 artists in the library').invert_yaxis()

## Connector tracks — songs that appear across many playlists
These recur across contexts; they are strong 'bridge' candidates and hint at
core taste anchors.

In [ ]:
connectors = q('''
  SELECT t.title, t.artist, COUNT(DISTINCT pt.playlist_id) AS in_playlists
  FROM tracks t JOIN playlist_tracks pt ON pt.track_id = t.id
  GROUP BY t.id ORDER BY in_playlists DESC LIMIT 15''')
connectors

## Takeaways
- Taste has visible **structure** even from metadata alone: dominant artists,
  recurring connector tracks, curated vs. dump playlists.
- These are the features a candidate generator would start from.
- Next: [`02-cooccurrence-and-taste.ipynb`](02-cooccurrence-and-taste.ipynb) turns
  'appears together' into a similarity signal.